# Fase 5 — Transformer geoespacial (atención sobre celdas)

Este notebook implementa la segunda arquitectura espacial del pipeline. Reutiliza el mismo preprocesado que el notebook 04 (catálogo USGS California+Alaska, declustering, rejilla 0.5°×0.5°, 5 características por celda, objetivo por celda) y la misma TCN compartida como bloque temporal; solo cambia el mecanismo de mezcla espacial.

| | GNN / GAT (nb 04) | Transformer geoespacial (nb 05) |
|---|---|---|
| Estructura espacial | grafo explícito (aristas por vecindad) | atención entre celdas (sin grafo) |
| Inyección del espacio | adyacencia de Haversine | codificación sinusoidal de la posición |
| Propagación | GCN / GAT sobre vecinos | self-attention enmascarada por región |

La capa de Proceso Gaussiano y los mapas de riesgo están en el notebook 06; el mejor modelo espacial alimenta esa fase.

## Estructura del notebook

1. Datos: catálogo y declustering.
2. Discretización espacial y características por celda.
3. Grafo de vecindad (define la máscara de atención).
4. Modelo y entrenamiento.
5. Evaluación y comparación con las referencias.

In [ ]:
# Entorno + NB_TAG (derivado automaticamente del NOMBRE de este notebook)
import os

try:
    from google.colab import drive

    # Si el import funciona, el entorno es Colab
    print("Entorno Colab detectado: Montando Google Drive...")
    estoy_drive = True
    drive.mount('/content/drive')

    import sys
    sys.path.append("/content/drive/MyDrive/VIU/TFM/code")
    sys.path.append("/content/drive/MyDrive/VIU/TFM/results")
    sys.path.append("/content/drive/MyDrive/VIU/TFM/USGS")

    # Nombre del notebook en Colab -> NB_TAG = primera parte antes del '_'
    from google.colab import _message
    _meta = _message.blocking_request("get_ipynb")["ipynb"]["metadata"]
    _nb_name = _meta.get("colab", {}).get("name", "05a.ipynb")
    NB_TAG = os.path.splitext(_nb_name)[0].split("_")[0]

except ImportError:
    estoy_drive = False
    # Entorno local (VS Code / Jupyter): nombre del fichero -> NB_TAG
    try:
        NB_TAG = os.path.splitext(os.path.basename(__vsc_ipynb_file__))[0].split("_")[0]
    except NameError:
        NB_TAG = "05a"

print(estoy_drive)
print(f"NB_TAG = {NB_TAG}")

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from tqdm import tqdm

!pip install -q seisbench

sys.path.append(os.path.abspath(""))

# Reproducibilidad (semillas fijas)
SEED = 42
np.random.seed(SEED)

## 1. Datos: catálogo y declustering

In [ ]:
# --- Configuración: lista de regiones (descarga concatenada desde USGS FDSN) ---
#
# Se admite una lista de regiones (California y Alaska): cada bbox se descarga por
# separado y se concatena. El grafo respeta los grupos geográficos (no se conectan
# celdas entre regiones distantes).

REGIONS = [
    {"name": "california", "minlatitude":  32.0, "maxlatitude":  42.0,
                           "minlongitude": -124.0, "maxlongitude": -114.0},
    {"name": "alaska",     "minlatitude":  51.0, "maxlatitude":  72.0,
                           "minlongitude": -180.0, "maxlongitude": -130.0},
]

# Otras posibles regiones (descomenta para añadir):
#  {"name": "japan",  "minlatitude":  24.0, "maxlatitude":  46.0, "minlongitude":  122.0, "maxlongitude":  146.0},
#  {"name": "italy",  "minlatitude":  35.0, "maxlatitude":  48.0, "minlongitude":    6.0, "maxlongitude":   19.0},
#  {"name": "chile",  "minlatitude": -45.0, "maxlatitude": -17.0, "minlongitude":  -76.0, "maxlongitude":  -66.0},

REGIONS_TAG       = "+".join(r["name"] for r in REGIONS)
STARTTIME         = "2000-01-01"
ENDTIME           = "2024-12-31"
MIN_MAG_CATALOGO  = 2.5
MIN_MAG_TARGET    = 4.5
GRID_DEG          = 0.5
PREDICTION_WINDOW = 30
LOOKBACK          = 60

# Bbox global para el grid (minimum enclosing box de las regiones)
REGION = {
    "minlatitude":  min(r["minlatitude"]  for r in REGIONS),
    "maxlatitude":  max(r["maxlatitude"]  for r in REGIONS),
    "minlongitude": min(r["minlongitude"] for r in REGIONS),
    "maxlongitude": max(r["maxlongitude"] for r in REGIONS),
}

# --- Rutas ---
if estoy_drive:
  # Rutas absolutas a Google Drive
  BASE_DIR = "/content/drive/MyDrive/VIU/TFM"

  RESULTS_DIR = os.path.join(BASE_DIR, "results")
  FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
  USGS_DIR    = os.path.join(BASE_DIR, "USGS")
else:
  RESULTS_DIR = os.path.join("..", "results")
  FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
  USGS_DIR    = os.path.join("..", "USGS")

for d in (RESULTS_DIR, FIGURES_DIR, USGS_DIR):
    os.makedirs(d, exist_ok = True)

# --- Cargar/Descargar catálogos por región y concatenar ---
import urllib.parse
import urllib.error

def fetch_region_catalog(region):
    """Carga del cache o descarga el catálogo USGS para una región."""
    path = os.path.join(USGS_DIR,
        f"{region['name']}_{STARTTIME[:4]}_{ENDTIME[:4]}_M{MIN_MAG_CATALOGO}.csv")
    if os.path.exists(path):
        print(f"  [{region['name']}] caché: {path}")
        df = pd.read_csv(path)
        df["time"] = pd.to_datetime(df["time"], format = "ISO8601", utc = True)
        return df

    print(f"  [{region['name']}] descargando desde USGS FDSN...")
    BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    fechas_start = pd.date_range(STARTTIME, ENDTIME, freq = "MS")
    fechas_end   = list(fechas_start[1:]) + [pd.to_datetime(ENDTIME) + pd.Timedelta(days = 1)]

    dfs = []
    for inicio, fin in tqdm(list(zip(fechas_start, fechas_end)),
                            desc = f"  USGS {region['name']}"):
        s, e = inicio.strftime("%Y-%m-%d"), fin.strftime("%Y-%m-%d")
        params = {
            "format": "csv", "starttime": s, "endtime": e,
            "minmagnitude": MIN_MAG_CATALOGO,
            "minlatitude":  region["minlatitude"],  "maxlatitude":  region["maxlatitude"],
            "minlongitude": region["minlongitude"], "maxlongitude": region["maxlongitude"],
            "orderby": "time-asc",
        }
        url = BASE_URL + "?" + urllib.parse.urlencode(params)
        try:
            df_chunk = pd.read_csv(url)
        except (pd.errors.EmptyDataError, urllib.error.HTTPError):
            df_chunk = pd.DataFrame()
        dfs.append(df_chunk)

    df = pd.concat(dfs, ignore_index = True)
    df["time"] = pd.to_datetime(df["time"], format = "ISO8601", utc = True)
    df = (df.drop_duplicates(subset = ["id"])
            .sort_values("time")
            .reset_index(drop = True))
    df.to_csv(path, index = False)
    print(f"  [{region['name']}] guardado: {path}")
    return df


print(f"Cargando {len(REGIONS)} región(es)...")
dfs_per_region = []
for region in REGIONS:
    df_r = fetch_region_catalog(region)
    df_r["region"] = region["name"]
    dfs_per_region.append(df_r)
    print(f"    └─ {region['name']}: {len(df_r):,} eventos M≥{MIN_MAG_CATALOGO}, "
          f"{(df_r['mag'] >= MIN_MAG_TARGET).sum()} M≥{MIN_MAG_TARGET}")

df_cat = pd.concat(dfs_per_region, ignore_index = True).sort_values("time").reset_index(drop = True)
df_cat["date"] = df_cat["time"].dt.floor("D")

print(f"\n=== Configuración ===")
print(f"Regiones:          {REGIONS_TAG}  ({len(REGIONS)} bbox)")
print(f"Bbox global:       lat [{REGION['minlatitude']}, {REGION['maxlatitude']}], "
      f"lon [{REGION['minlongitude']}, {REGION['maxlongitude']}]")
print(f"Ventana temporal:  {STARTTIME} → {ENDTIME}")
print(f"M catálogo ≥ {MIN_MAG_CATALOGO}  |  M objetivo ≥ {MIN_MAG_TARGET}")
print(f"Grid espacial:     {GRID_DEG}° × {GRID_DEG}°  ({int((REGION['maxlatitude']-REGION['minlatitude'])/GRID_DEG)} × "
      f"{int((REGION['maxlongitude']-REGION['minlongitude'])/GRID_DEG)} celdas máximas)")
print(f"PREDICTION_WINDOW: {PREDICTION_WINDOW} días")
print(f"LOOKBACK:          {LOOKBACK} días")
print()
print(f"=== Catálogo combinado ===")
print(f"Total eventos:     {len(df_cat):,}")
print(f"Rango temporal:    {df_cat['time'].min()} → {df_cat['time'].max()}")
print(f"Rango magnitud:    {df_cat['mag'].min():.1f} → {df_cat['mag'].max():.1f}")
print(f"Eventos M ≥ {MIN_MAG_TARGET}:  {(df_cat['mag'] >= MIN_MAG_TARGET).sum():,}  "
      f"(~{(df_cat['mag'] >= MIN_MAG_TARGET).sum() / 15.0:.1f}/año)")
print()
print(f"Por región:")
print(df_cat.groupby("region").agg(
    n_total      = ("mag", "size"),
    n_M_target   = ("mag", lambda m: (m >= MIN_MAG_TARGET).sum()),
).to_string())

In [ ]:
# --- Declustering de Gardner-Knopoff (1974) ---
# Identifica mainshocks INDEPENDIENTES (filtra aftershocks/foreshocks dependientes).
# Tras el filtrado quedan los mainshocks independientes, no las secuencias de réplicas.
#
# Ventanas (van Stiphout et al. 2012):
#   L(km)   = 10^(0.1238·M + 0.983)
#   T(días) = 10^(0.5409·M - 0.547)        si M ≤ 6.5
#   T(días) = 10^(0.032·M + 2.7389)        si M >  6.5

def gardner_knopoff_window(M):
    R_km = 10 ** (0.1238 * M + 0.983)
    if M <= 6.5:
        T_days = 10 ** (0.5409 * M - 0.547)
    else:
        T_days = 10 ** (0.032 * M + 2.7389)
    return T_days, R_km

def haversine_km(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371.0
    lat1r = np.radians(lat1)
    lat2r = np.radians(lat2_arr)
    dlat  = lat2r - lat1r
    dlon  = np.radians(lon2_arr - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df_target_pool = (df_cat[df_cat["mag"] >= MIN_MAG_TARGET]
                    .sort_values("time")
                    .reset_index(drop = True))
times_sec = (df_target_pool["time"] - df_target_pool["time"].iloc[0]).dt.total_seconds().values
lats      = df_target_pool["latitude"].values
lons      = df_target_pool["longitude"].values
mags      = df_target_pool["mag"].values
n         = len(df_target_pool)

is_cluster  = np.zeros(n, dtype = bool)
sort_by_mag = np.argsort(-mags)

for i in sort_by_mag:
    if is_cluster[i]:
        continue
    M_main       = mags[i]
    T_days, R_km = gardner_knopoff_window(M_main)
    T_sec        = T_days * 86400.0
    dt           = np.abs(times_sec - times_sec[i])
    candidates   = np.where((dt <= T_sec) & (dt > 0) & (mags < M_main) & (~is_cluster))[0]
    if len(candidates) == 0:
        continue
    dist        = haversine_km(lats[i], lons[i], lats[candidates], lons[candidates])
    cluster_idx = candidates[dist <= R_km]
    is_cluster[cluster_idx] = True

df_targets = df_target_pool[~is_cluster].copy()
df_targets["date"] = df_targets["time"].dt.floor("D")

print(f"=== Declustering Gardner-Knopoff (1974) ===")
print(f"Eventos M ≥ {MIN_MAG_TARGET} originales:        {n}")
print(f"Mainshocks independientes:        {len(df_targets)}  ({100 * len(df_targets) / n:.1f}%)")
print(f"Aftershocks/foreshocks eliminados: {n - len(df_targets)}")
print(f"Tasa anual de mainshocks:         {len(df_targets) / 15.0:.2f}/año")
print()
print(f"(Debe coincidir con los 71 mainshocks reportados por el nb 03)")

## 2. Discretización espacial y características

In [ ]:
# --- Discretización espacial: grid 0.5° × 0.5° y filtrado de celdas activas ---
# Cada evento se asigna a una celda. Las celdas con muy pocos eventos se descartan
# (más ruido que señal).
# También se asigna a cada celda su REGIÓN (california/alaska) para mantener la
# separación lógica en el grafo y el cálculo de b-value.

# Crear ejes del grid sobre el bbox GLOBAL
lat_edges = np.arange(REGION["minlatitude"],  REGION["maxlatitude"]  + 1e-9, GRID_DEG)
lon_edges = np.arange(REGION["minlongitude"], REGION["maxlongitude"] + 1e-9, GRID_DEG)
n_lat = len(lat_edges) - 1
n_lon = len(lon_edges) - 1

# Asignar cada evento del catálogo a su celda
df_cat["lat_idx"] = np.clip(np.digitize(df_cat["latitude"],  lat_edges) - 1, 0, n_lat - 1)
df_cat["lon_idx"] = np.clip(np.digitize(df_cat["longitude"], lon_edges) - 1, 0, n_lon - 1)
df_cat["cell_id"] = df_cat["lat_idx"] * n_lon + df_cat["lon_idx"]

# El declustering ya generó df_targets; se replica la asignación a celda
df_targets["lat_idx"] = np.clip(np.digitize(df_targets["latitude"],  lat_edges) - 1, 0, n_lat - 1)
df_targets["lon_idx"] = np.clip(np.digitize(df_targets["longitude"], lon_edges) - 1, 0, n_lon - 1)
df_targets["cell_id"] = df_targets["lat_idx"] * n_lon + df_targets["lon_idx"]

# --- Filtrado de celdas activas ---
MIN_EVENTOS_CELDA = 30

events_per_cell = df_cat.groupby("cell_id").size()
active_cells    = events_per_cell[events_per_cell >= MIN_EVENTOS_CELDA].index.values
active_cells    = np.sort(active_cells)
n_active        = len(active_cells)

cell_id_to_idx = {cid: i for i, cid in enumerate(active_cells)}

# --- Asignar cada celda activa a su región (la que mayoritariamente contiene sus eventos) ---
def cell_region(cell_id):
    """Devuelve el nombre de la región que contiene los eventos de esta celda."""
    region_counts = df_cat[df_cat["cell_id"] == cell_id]["region"].value_counts()
    return region_counts.index[0] if len(region_counts) else None

cell_region_arr = np.array([cell_region(cid) for cid in active_cells])

# Sanity check: cada celda activa debe tener al menos una región
assert all(r is not None for r in cell_region_arr), "Hay celdas activas sin región asignada"

# Mainshocks dentro vs fuera de celdas activas
mainshocks_active  = df_targets[df_targets["cell_id"].isin(active_cells)].copy()
mainshocks_outside = df_targets[~df_targets["cell_id"].isin(active_cells)].copy()

print(f"=== Grid espacial ===")
print(f"Bbox global:     lat [{REGION['minlatitude']}, {REGION['maxlatitude']}], "
      f"lon [{REGION['minlongitude']}, {REGION['maxlongitude']}]")
print(f"Tamaño del grid: {n_lat} × {n_lon} = {n_lat * n_lon} celdas máximas")
print(f"Celdas con ≥ 1 evento:                {(events_per_cell > 0).sum()}")
print(f"Celdas activas (≥ {MIN_EVENTOS_CELDA} eventos M≥{MIN_MAG_CATALOGO}):  {n_active}")
print()
print(f"Distribución de celdas activas por región:")
unique, counts = np.unique(cell_region_arr, return_counts = True)
for r, c in zip(unique, counts):
    print(f"  {r:12s}  {c} celdas")
print()
print(f"=== Cobertura de mainshocks ===")
print(f"Mainshocks totales (post G-K):        {len(df_targets)}")
print(f"  ├─ dentro de celdas activas:        {len(mainshocks_active)}  "
      f"({100*len(mainshocks_active)/len(df_targets):.1f}%)")
print(f"  └─ fuera (en celdas con baja sismicidad): {len(mainshocks_outside)}")
print()
print(f"Distribución de eventos en celdas activas:")
print(f"  median: {int(np.median(events_per_cell[active_cells]))} eventos/celda")
print(f"  min:    {int(np.min(events_per_cell[active_cells]))} eventos/celda")
print(f"  max:    {int(np.max(events_per_cell[active_cells]))} eventos/celda")
print()
if len(mainshocks_outside) > 0:
    print(f"⚠️  Mainshocks fuera de celdas activas (primeros 10):")
    print(mainshocks_outside[["time", "latitude", "longitude", "mag", "place"]].head(10).to_string(index = False))

In [ ]:
# --- Figura: estructura espacial del grid (heatmap + mainshocks) — 300 DPI ---
import matplotlib.patches as mpatches
from matplotlib.colors import LogNorm

fig, ax = plt.subplots(1, 1, figsize = (11, 11))

# Heatmap de actividad por celda (log scale)
heat = np.full((n_lat, n_lon), np.nan)
for cid, cnt in events_per_cell.items():
    li, lo = cid // n_lon, cid % n_lon
    heat[li, lo] = cnt

im = ax.imshow(
    heat,
    origin    = "lower",
    extent    = [REGION["minlongitude"], REGION["maxlongitude"],
                 REGION["minlatitude"],  REGION["maxlatitude"]],
    cmap      = "YlOrRd",
    norm      = LogNorm(vmin = 1, vmax = max(events_per_cell)),
    alpha     = 0.85,
    aspect    = "auto",
)
plt.colorbar(im, ax = ax, label = f"Nº eventos M≥{MIN_MAG_CATALOGO} (log)", shrink = 0.7)

# Bordes del grid
for la in lat_edges:
    ax.axhline(la, color = "gray", linewidth = 0.2, alpha = 0.4)
for lo in lon_edges:
    ax.axvline(lo, color = "gray", linewidth = 0.2, alpha = 0.4)

# Borde grueso para celdas ACTIVAS
for cid in active_cells:
    li, lo = cid // n_lon, cid % n_lon
    rect = mpatches.Rectangle(
        (lon_edges[lo], lat_edges[li]),
        GRID_DEG, GRID_DEG,
        linewidth = 1.0, edgecolor = "navy", facecolor = "none", alpha = 0.7,
    )
    ax.add_patch(rect)

# Mainshocks dentro de celdas activas
ax.scatter(
    mainshocks_active["longitude"], mainshocks_active["latitude"],
    s = 60 + 25 * (mainshocks_active["mag"] - MIN_MAG_TARGET),
    c = "darkred", marker = "*", edgecolor = "black", linewidth = 0.6, zorder = 5,
    label = f"Mainshocks dentro ({len(mainshocks_active)})",
)

# Mainshocks fuera (zonas no cubiertas por el grafo)
ax.scatter(
    mainshocks_outside["longitude"], mainshocks_outside["latitude"],
    s = 80, c = "gray", marker = "X", edgecolor = "black", linewidth = 0.6, zorder = 5,
    label = f"Mainshocks fuera ({len(mainshocks_outside)})",
)

ax.set_xlim(REGION["minlongitude"], REGION["maxlongitude"])
ax.set_ylim(REGION["minlatitude"],  REGION["maxlatitude"])
ax.set_xlabel("Longitud (°)")
ax.set_ylabel("Latitud (°)")
ax.set_title(
    f"Estructura espacial — California+Alaska {STARTTIME[:4]}–{ENDTIME[:4]}\n"
    f"{n_active} celdas activas (≥{MIN_EVENTOS_CELDA} ev) · {len(mainshocks_active)}/{len(df_targets)} mainshocks cubiertos",
    fontweight = "bold",
)

# Leyenda con celdas activas y mainshocks
active_patch = mpatches.Patch(facecolor = "none", edgecolor = "navy", linewidth = 1.5,
                              label = f"Celda activa ({n_active})")
ax.legend(handles = [active_patch] + ax.get_legend_handles_labels()[0],
          loc = "lower left", framealpha = 0.95)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR,
            f"{NB_TAG}_grid_espacial_M{MIN_MAG_TARGET}_min{MIN_EVENTOS_CELDA}.png"),
            dpi = 300, bbox_inches = "tight")
plt.show()

In [ ]:
# --- Features espaciotemporales por celda y día (5 canales) ---
# Canales:
#   1. log_count       — log(1 + nº eventos M≥Mc en celda y día)         LOCAL
#   2. log_energy      — log(1 + energía Kanamori en celda y día)        LOCAL
#   3. max_mag         — magnitud máxima del día en la celda             LOCAL
#   4. log_count_M35   — log(1 + nº eventos M≥3.5 en celda y día)        LOCAL
#   5. b_value_30d     — b-value móvil 30 días POR REGIÓN                GLOBAL_REGIONAL
#                        Cada celda recibe el b-value calculado SOLO con eventos
#                        de su misma región (California o Alaska son tectónicamente
#                        muy distintas, mezclar contaminaría la señal).

date_range = pd.date_range(STARTTIME, ENDTIME, freq = "D", tz = "UTC")
n_days     = len(date_range)
date_to_idx = {d: i for i, d in enumerate(date_range)}

# Sólo eventos en celdas activas
df_cat_active = df_cat[df_cat["cell_id"].isin(active_cells)].copy()
df_cat_active["energy"]   = 10 ** (1.5 * df_cat_active["mag"] + 4.8)
df_cat_active["day_idx"]  = df_cat_active["date"].map(date_to_idx)
df_cat_active["cell_idx"] = df_cat_active["cell_id"].map(cell_id_to_idx)

# Tensor (n_days, n_active, 5)
n_features = 5
X_full = np.zeros((n_days, n_active, n_features), dtype = np.float32)

g = df_cat_active.groupby(["day_idx", "cell_idx"])

# Canal 0: log_count
v = g.size().reset_index(name = "v")
X_full[v["day_idx"], v["cell_idx"], 0] = np.log1p(v["v"].values)

# Canal 1: log_energy
v = g["energy"].sum().reset_index(name = "v")
X_full[v["day_idx"], v["cell_idx"], 1] = np.log1p(v["v"].values)

# Canal 2: max_mag
v = g["mag"].max().reset_index(name = "v")
X_full[v["day_idx"], v["cell_idx"], 2] = v["v"].values

# Canal 3: log_count_M35
df35 = df_cat_active[df_cat_active["mag"] >= 3.5]
v = df35.groupby(["day_idx", "cell_idx"]).size().reset_index(name = "v")
X_full[v["day_idx"], v["cell_idx"], 3] = np.log1p(v["v"].values)

# --- Canal 4: b-value 30 días POR REGIÓN ---
WINDOW_BVAL = 30

def b_value_serie(df_region, date_range, window):
    """Serie de b-value móvil para los eventos de UNA región concreta."""
    mags_per_day = (df_region[df_region["mag"] >= MIN_MAG_CATALOGO]
                      .groupby("date")["mag"].apply(list)
                      .reindex(date_range)
                      .apply(lambda x: x if isinstance(x, list) else []))
    b = np.full(len(date_range), np.nan, dtype = np.float32)
    for i in range(len(date_range)):
        lo = max(0, i - window + 1)
        mags_win = [m for sub in mags_per_day.iloc[lo:i+1] for m in sub]
        if len(mags_win) >= 20:
            b[i] = np.log10(np.e) / (np.mean(mags_win) - (MIN_MAG_CATALOGO - 0.05))
    return pd.Series(b, index = date_range).ffill().fillna(1.0).values

b_value_per_region = {}
for region_name in [r["name"] for r in REGIONS]:
    df_region = df_cat[df_cat["region"] == region_name]
    b_value_per_region[region_name] = b_value_serie(df_region, date_range, WINDOW_BVAL)
    print(f"  b-value {region_name:12s}  mean={np.mean(b_value_per_region[region_name]):.3f}  "
          f"std={np.std(b_value_per_region[region_name]):.3f}  "
          f"rango=[{np.min(b_value_per_region[region_name]):.2f}, "
          f"{np.max(b_value_per_region[region_name]):.2f}]")

# Asignar el b-value de la región correspondiente a cada celda
for ci in range(n_active):
    region_name = cell_region_arr[ci]
    X_full[:, ci, 4] = b_value_per_region[region_name]

print()
print(f"=== Features espaciotemporales ===")
print(f"Tensor X_full: shape {X_full.shape}  (n_dias, n_celdas_activas, n_canales)")
print(f"Canales: log_count, log_energy, max_mag, log_count_M35, b_value_30d (regional)")
print()
print(f"Densidad (% entradas con valor > 0):")
print(f"  log_count:     {(X_full[:, :, 0] > 0).mean() * 100:5.2f}%")
print(f"  log_energy:    {(X_full[:, :, 1] > 0).mean() * 100:5.2f}%")
print(f"  max_mag:       {(X_full[:, :, 2] > 0).mean() * 100:5.2f}%")
print(f"  log_count_M35: {(X_full[:, :, 3] > 0).mean() * 100:5.2f}%")
print(f"  b_value_30d:   100.00% (regional)")
print()
print(f"Estadísticos por canal (sobre TODAS las entradas):")
for j, name in enumerate(["log_count", "log_energy", "max_mag", "log_count_M35", "b_value_30d"]):
    arr = X_full[:, :, j]
    print(f"  {name:14s}  mean={arr.mean():>7.3f}  std={arr.std():>7.3f}  "
          f"min={arr.min():>6.2f}  max={arr.max():>7.2f}")

In [ ]:
# --- Target binario POR CELDA con expansión espacial ---
# Definición del objetivo por celda:
#   y[t, c] = 1 si hay un mainshock M ≥ MIN_MAG_TARGET en la celda c o en sus
#               vecinas inmediatas (3×3 box) durante los próximos PREDICTION_WINDOW días.
#
# Por qué expansión espacial 3×3 (≈ 50 km de radio):
#   1. Aumenta la proporción de positivos (sin ella sería ~1% global, muy desbalanceada).
#   2. Modela la incertidumbre de localización (epicentro USGS tiene error de varios km).
#   3. Tiene sentido físico: "habrá M≥4.5 cerca de aquí en próx. 30 días".
#   4. Es la formulación estándar en estudios de prediccion sísmica (Field et al. 2014).

NEIGHBOR_RADIUS = 1   # 1 celda de radio → 3×3 box centrada (≈ 50 km)

# Pre-computar para cada mainshock: lista de cell_idx (compactos) afectados
def expand_to_neighbors(lat_idx, lon_idx, radius = NEIGHBOR_RADIUS):
    """Devuelve los cell_idx compactos de la celda (lat_idx, lon_idx) y sus vecinas."""
    affected = []
    for dlat in range(-radius, radius + 1):
        for dlon in range(-radius, radius + 1):
            li, lo = lat_idx + dlat, lon_idx + dlon
            if 0 <= li < n_lat and 0 <= lo < n_lon:
                cid = li * n_lon + lo
                if cid in cell_id_to_idx:
                    affected.append(cell_id_to_idx[cid])
    return affected

# Tensor (n_days, n_active) de "este día tiene mainshock impactando esta celda"
mainshock_at = np.zeros((n_days, n_active), dtype = bool)
for _, m in df_targets.iterrows():
    day_idx = date_to_idx.get(m["date"])
    if day_idx is None:
        continue
    affected_idxs = expand_to_neighbors(int(m["lat_idx"]), int(m["lon_idx"]))
    for ci in affected_idxs:
        mainshock_at[day_idx, ci] = True

# y_full[t, c] = 1 si en (t+1, ..., t+PREDICTION_WINDOW) hay mainshock impactando c
y_full = np.zeros((n_days, n_active), dtype = np.int32)
for t in range(n_days - PREDICTION_WINDOW):
    y_full[t] = mainshock_at[t + 1 : t + 1 + PREDICTION_WINDOW].any(axis = 0).astype(np.int32)

print(f"=== Target binario por celda ===")
print(f"y_full shape: {y_full.shape}  (n_dias, n_celdas_activas)")
print(f"Expansión espacial: 3×3 celdas (radio {NEIGHBOR_RADIUS}, ~{int(GRID_DEG * 111)} km)")
print(f"PREDICTION_WINDOW: {PREDICTION_WINDOW} días")
print()
print(f"Total entradas (día × celda):       {y_full.size:,}")
print(f"Entradas con y=1:                   {y_full.sum():,}  ({y_full.mean() * 100:.2f}%)")
print(f"Días con al menos 1 celda y=1:      {(y_full.sum(axis = 1) > 0).sum():,}  "
      f"({(y_full.sum(axis = 1) > 0).mean() * 100:.1f}%)")
print(f"Celdas con al menos 1 día y=1:      {(y_full.sum(axis = 0) > 0).sum()} / {n_active}")
print()
print(f"Distribución de positivos por celda (días con y=1):")
pos_per_cell = y_full.sum(axis = 0)
print(f"  median: {int(np.median(pos_per_cell))}  "
      f"max: {int(pos_per_cell.max())}  "
      f"con 0 positivos: {(pos_per_cell == 0).sum()} celdas")

## 3. Grafo de vecindad y máscara de atención

In [ ]:
# --- Construcción del grafo: vecindad geográfica DENTRO DE CADA REGIÓN ---
# California y Alaska están a >3000 km, por lo que el grafo se restringe a vecinos
# de la misma región. Define dos componentes conexas (una por región).

NEIGHBOR_DIST_KM = 100      # threshold geográfico (intra-región)
K_FALLBACK       = 2        # mínimo de vecinos garantizado por KNN intra-región

# Centroides de las celdas activas
cell_centroids = np.zeros((n_active, 2), dtype = np.float32)
for ci, cid in enumerate(active_cells):
    li, lo = cid // n_lon, cid % n_lon
    cell_centroids[ci, 0] = (lat_edges[li] + lat_edges[li + 1]) / 2
    cell_centroids[ci, 1] = (lon_edges[lo] + lon_edges[lo + 1]) / 2

def haversine_matrix(coords):
    R = 6371.0
    lats = np.radians(coords[:, 0])[:, None]
    lons = np.radians(coords[:, 1])[:, None]
    dlat = lats - lats.T
    dlon = lons - lons.T
    a = np.sin(dlat / 2) ** 2 + np.cos(lats) * np.cos(lats.T) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

dist_matrix = haversine_matrix(cell_centroids)

# Máscara de "misma región": True si i,j son de la misma región
same_region = (cell_region_arr[:, None] == cell_region_arr[None, :])

# Adyacencia por threshold geográfico, RESTRINGIDA a la misma región
adj_matrix = ((dist_matrix <= NEIGHBOR_DIST_KM) & (dist_matrix > 0) & same_region).astype(np.float32)

# Fallback KNN: para nodos con grado < K_FALLBACK, conectar con sus K vecinos
# más cercanos DENTRO DE SU REGIÓN (no entre regiones)
n_fallback_added = 0
for i in range(n_active):
    if adj_matrix[i].sum() < K_FALLBACK:
        same_idx  = np.where(same_region[i] & (np.arange(n_active) != i))[0]
        if len(same_idx) == 0:
            continue
        order_in_region = same_idx[np.argsort(dist_matrix[i, same_idx])]
        for j in order_in_region[:K_FALLBACK]:
            if adj_matrix[i, j] == 0:
                adj_matrix[i, j] = 1
                adj_matrix[j, i] = 1
                n_fallback_added += 1

# --- Stats del grafo ---
n_edges = int(adj_matrix.sum()) // 2
degrees = adj_matrix.sum(axis = 1)

print(f"=== Grafo de celdas (intra-región) ===")
print(f"Nodos:                    {n_active}")
print(f"Aristas (no dirigidas):   {n_edges}")
print(f"Distancia geográfica:     ≤ {NEIGHBOR_DIST_KM} km  (+ KNN fallback k={K_FALLBACK}, intra-región)")
print(f"Aristas añadidas por KNN: {n_fallback_added}")
print()
print(f"Grado de los nodos:")
print(f"  median:  {int(np.median(degrees))}")
print(f"  mean:    {degrees.mean():.1f}")
print(f"  min:     {int(degrees.min())}")
print(f"  max:     {int(degrees.max())}")
print(f"  isolated (grado 0): {int((degrees == 0).sum())}")
print()
print(f"Densidad del grafo: {n_edges / (n_active * (n_active - 1) / 2) * 100:.2f}%")
print()
print(f"Componentes por región (cada región es componente independiente):")
for r_name in [r["name"] for r in REGIONS]:
    n_r = (cell_region_arr == r_name).sum()
    n_r_edges = int(adj_matrix[cell_region_arr == r_name][:, cell_region_arr == r_name].sum() // 2)
    print(f"  {r_name:12s}  {n_r} nodos, {n_r_edges} aristas")

In [ ]:
# --- Figura: visualización del grafo sobre el mapa — 300 DPI ---
fig, ax = plt.subplots(1, 1, figsize = (11, 11))

# Heatmap de fondo (mismo que cell-5 pero más tenue para no distraer del grafo)
heat = np.full((n_lat, n_lon), np.nan)
for cid, cnt in events_per_cell.items():
    li, lo = cid // n_lon, cid % n_lon
    heat[li, lo] = cnt

ax.imshow(
    heat,
    origin    = "lower",
    extent    = [REGION["minlongitude"], REGION["maxlongitude"],
                 REGION["minlatitude"],  REGION["maxlatitude"]],
    cmap      = "YlOrRd",
    norm      = LogNorm(vmin = 1, vmax = max(events_per_cell)),
    alpha     = 0.35,    # más tenue que el cell-5
    aspect    = "auto",
)

# --- Aristas del grafo ---
edge_lines = []
for i in range(n_active):
    for j in range(i + 1, n_active):
        if adj_matrix[i, j] > 0:
            edge_lines.append([
                (cell_centroids[i, 1], cell_centroids[i, 0]),
                (cell_centroids[j, 1], cell_centroids[j, 0]),
            ])
from matplotlib.collections import LineCollection
ax.add_collection(LineCollection(edge_lines, colors = "navy", linewidths = 0.6, alpha = 0.45))

# --- Nodos del grafo (tamaño proporcional al grado, color = positivos en y_full) ---
pos_per_cell = y_full.sum(axis = 0)   # nº días con target=1 por celda
sc = ax.scatter(
    cell_centroids[:, 1], cell_centroids[:, 0],
    s = 30 + 8 * degrees,              # tamaño = grado en el grafo
    c = pos_per_cell,
    cmap = "Blues",
    vmin = 0, vmax = pos_per_cell.max(),
    edgecolor = "navy", linewidth = 0.6, zorder = 5,
)
plt.colorbar(sc, ax = ax, label = "Días con y=1 por celda", shrink = 0.7)

# Mainshocks dentro de celdas activas (estrellas rojas)
ax.scatter(
    mainshocks_active["longitude"], mainshocks_active["latitude"],
    s = 50 + 20 * (mainshocks_active["mag"] - MIN_MAG_TARGET),
    c = "darkred", marker = "*", edgecolor = "black", linewidth = 0.5, zorder = 6,
    label = f"Mainshock M≥{MIN_MAG_TARGET} ({len(mainshocks_active)})",
)

ax.set_xlim(REGION["minlongitude"], REGION["maxlongitude"])
ax.set_ylim(REGION["minlatitude"],  REGION["maxlatitude"])
ax.set_xlabel("Longitud (°)")
ax.set_ylabel("Latitud (°)")
ax.set_title(
    f"Grafo de celdas activas — California+Alaska {STARTTIME[:4]}–{ENDTIME[:4]}\n"
    f"{n_active} nodos, {n_edges} aristas (≤{NEIGHBOR_DIST_KM} km + KNN k={K_FALLBACK}) · "
    f"grado medio {degrees.mean():.1f}",
    fontweight = "bold",
)
ax.legend(loc = "lower left", framealpha = 0.95)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR,
            f"{NB_TAG}_grafo_celdas_d{NEIGHBOR_DIST_KM}km.png"),
            dpi = 300, bbox_inches = "tight")
plt.show()

## 4. Modelo y entrenamiento

In [ ]:
# --- Definición del Transformer geoespacial: TCN compartido + atención sobre celdas ---
import importlib
import torch
import utils.modelos
importlib.reload(utils.modelos)   # fuerza recarga (caché de Jupyter)
from utils.modelos import SpatioTemporalTransformer, geospatial_encoding, region_attention_mask

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
print()

# Hiperparámetros del modelo
NUM_CHANNELS = (32, 32, 64)        # 3 niveles del TCN (dilations 1·2·4 → RF=29)
KERNEL_SIZE  = 3
D_MODEL         = 128                  # dim del embedding del Transformer (= salida TCN)
N_HEADS      = 4                   # cabezas de atención
N_LAYERS     = 2                   # capas del TransformerEncoder
DIM_FF          = 256                 # dim del feed-forward interno
DROPOUT         = 0.20

# Receptive field temporal del TCN
n_layers_tcn    = len(NUM_CHANNELS)
receptive_field = 1 + 2 * (KERNEL_SIZE - 1) * (2 ** n_layers_tcn - 1)

# --- Codificación geoespacial sinusoidal de los centroides (lat, lon) ---
# Sustituye al grafo de la GNN: el "dónde" de cada celda se inyecta como una
# positional encoding 2D que se suma a los embeddings antes de la atención.
# Se reutiliza el nombre A_norm (segundo argumento del forward) para no tocar las
# celdas de entrenamiento/evaluación; aquí NO es una adyacencia, es la geo-encoding.
# --- Positional Encoding usando Índices Enteros de la Cuadrícula ---
# Se usan los índices enteros de rejilla (lat_idx, lon_idx) en lugar de (latitud,
# longitud) para que las frecuencias sinusoidales se ajusten a la escala del modelo.
cell_indices = np.zeros((n_active, 2), dtype=np.float32)
for ci, cid in enumerate(active_cells):
    li, lo = cid // n_lon, cid % n_lon
    cell_indices[ci, 0] = li
    cell_indices[ci, 1] = lo

A_norm = geospatial_encoding(cell_indices, D_MODEL).to(DEVICE)

# Construir la máscara de atención basada en distancia (<= 100km) para evitar "Low-Pass Filtering"
# adj_matrix vale 1 si están conectados (y en misma región), 0 si no.
# La máscara requiere True para BLOQUEAR atención.
import numpy as np
import torch
mask_np = (adj_matrix + np.eye(n_active)) == 0
distance_mask = torch.from_numpy(mask_np).to(DEVICE)


def build_model(dropout = DROPOUT):
    """Construye el Transformer geoespacial. Reutilizado por el entrenamiento."""
    return SpatioTemporalTransformer(
        n_inputs        = n_features,
        num_channels    = NUM_CHANNELS,
        kernel_size     = KERNEL_SIZE,
        d_model         = D_MODEL,
        n_heads         = N_HEADS,
        n_layers        = N_LAYERS,
        dropout         = dropout,
        dim_feedforward = DIM_FF,
        attn_mask       = distance_mask,
    ).to(DEVICE)


# --- Instanciar el modelo (para inspección / smoke test) ---
torch.manual_seed(SEED)
model = build_model(DROPOUT)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"=== SpatioTemporalTransformer (geoespacial) ===")
print(f"Inputs por celda:    {n_features} canales × {LOOKBACK} días")
print(f"TCN canales:         {NUM_CHANNELS}  (RF temporal = {receptive_field} días)")
print(f"d_model:             {D_MODEL}  |  cabezas: {N_HEADS}  |  capas: {N_LAYERS}  |  ff: {DIM_FF}")
print(f"Dropout:             {DROPOUT}")
print(f"Parámetros:          {n_params:,}")
print(f"Geo-encoding (pe):   shape {tuple(A_norm.shape)}  (n_celdas, d_model)  [reemplaza al grafo]")
print()
print(model)
print()

# --- Smoke test ---
with torch.no_grad():
    x_dummy = torch.randn(4, n_active, LOOKBACK, n_features).to(DEVICE)
    out_dummy = model(x_dummy, A_norm)
print(f"Smoke test forward:")
print(f"  input:   {tuple(x_dummy.shape)}  (batch, n_nodes, lookback, n_features)")
print(f"  output:  {tuple(out_dummy.shape)}  (batch, n_nodes) logits")
print(f"  sample logits:  {out_dummy[0, :5].cpu().numpy().round(3)}  (primeros 5 nodos del batch 0)")

In [ ]:
# --- Splits temporales + normalización + Dataset perezoso ---
from torch.utils.data import Dataset, DataLoader

TRAIN_END = "2020-12-31"
VAL_END   = "2022-12-31"

# Día válido: t debe tener LOOKBACK días de historia + PREDICTION_WINDOW de futuro
valid_t = np.arange(LOOKBACK, n_days - PREDICTION_WINDOW)
dates_seq = date_range[valid_t]

# --- Split temporal ---
train_mask = dates_seq <= pd.Timestamp(TRAIN_END, tz = "UTC")
val_mask   = (dates_seq > pd.Timestamp(TRAIN_END, tz = "UTC")) & \
             (dates_seq <= pd.Timestamp(VAL_END, tz = "UTC"))
test_mask  = dates_seq > pd.Timestamp(VAL_END, tz = "UTC")

t_train = valid_t[train_mask]
t_val   = valid_t[val_mask]
t_test  = valid_t[test_mask]

# --- Normalización z-score con stats SÓLO de los días de train ---
# Estadísticos sobre el área cubierta por las ventanas de train (días desde
# t_train.min()-LOOKBACK hasta t_train.max()).
train_window_lo = int(t_train.min()) - LOOKBACK
train_window_hi = int(t_train.max())
flat_train = X_full[train_window_lo:train_window_hi].reshape(-1, n_features)   # (T·N, F)

mean_train = flat_train.mean(axis = 0)
std_train  = flat_train.std (axis = 0) + 1e-8

# Normalizar TODO X_full una vez (broadcasting sobre dimensiones de día y celda)
X_norm = (X_full - mean_train) / std_train
X_norm = X_norm.astype(np.float32)

# pos_weight: ratio neg/pos en TRAIN (todas las celdas, todos los días de train)
y_train_flat = y_full[t_train].flatten()
POS_WEIGHT = float((y_train_flat == 0).sum() / max(y_train_flat.sum(), 1))


class SpatioTemporalDataset(Dataset):
    """
    Dataset perezoso: extrae ventana (n_nodes, lookback, n_features) y target (n_nodes,)
    en cada __getitem__, sin materializar el tensor (n_samples, n_nodes, lookback, n_features).
    """
    def __init__(self, X_norm, y_full, valid_t, lookback):
        self.X       = X_norm        # (n_days, n_nodes, n_features)
        self.y       = y_full        # (n_days, n_nodes)
        self.valid_t = valid_t
        self.lookback = lookback

    def __len__(self):
        return len(self.valid_t)

    def __getitem__(self, idx):
        t = int(self.valid_t[idx])
        # X[t-LOOKBACK:t]: (lookback, n_nodes, n_features) → permutar a (n_nodes, lookback, n_features)
        x = self.X[t - self.lookback : t].transpose(1, 0, 2).copy()
        y = self.y[t].astype(np.float32)
        return torch.from_numpy(x), torch.from_numpy(y)


train_ds = SpatioTemporalDataset(X_norm, y_full, t_train, LOOKBACK)
val_ds   = SpatioTemporalDataset(X_norm, y_full, t_val,   LOOKBACK)
test_ds  = SpatioTemporalDataset(X_norm, y_full, t_test,  LOOKBACK)

print(f"=== Splits temporales ===")
print(f"LOOKBACK={LOOKBACK}  PREDICTION_WINDOW={PREDICTION_WINDOW}")
print(f"Train: {dates_seq[train_mask].min().date()} → {dates_seq[train_mask].max().date()}  "
      f"|  {len(train_ds):>5} muestras  |  pos = {y_full[t_train].sum():>5}  "
      f"({y_full[t_train].mean()*100:.2f}% celdas-días)")
print(f"Val:   {dates_seq[val_mask].min().date()} → {dates_seq[val_mask].max().date()}  "
      f"|  {len(val_ds):>5} muestras  |  pos = {y_full[t_val].sum():>5}  "
      f"({y_full[t_val].mean()*100:.2f}% celdas-días)")
print(f"Test:  {dates_seq[test_mask].min().date()} → {dates_seq[test_mask].max().date()}  "
      f"|  {len(test_ds):>5} muestras  |  pos = {y_full[t_test].sum():>5}  "
      f"({y_full[t_test].mean()*100:.2f}% celdas-días)")
print()
print(f"Stats z-score (mean/std por canal, calculadas con TRAIN):")
for j, name in enumerate(["log_count", "log_energy", "max_mag", "log_count_M35", "b_value_30d"]):
    print(f"  {name:14s}  mean = {mean_train[j]:>9.4f}   std = {std_train[j]:>9.4f}")
print()
print(f"pos_weight para BCE (n_neg/n_pos en train): {POS_WEIGHT:.2f}")
print()

# Smoke test del Dataset
x_sample, y_sample = train_ds[0]
print(f"Sample del Dataset:")
print(f"  x shape: {tuple(x_sample.shape)}  dtype={x_sample.dtype}")
print(f"  y shape: {tuple(y_sample.shape)}  dtype={y_sample.dtype}  (positivos: {int(y_sample.sum())})")

In [ ]:
# --- Función de entrenamiento + smoke test ---
import random
import copy
import time
from sklearn.metrics import roc_auc_score

# NB_TAG viene de la celda inicial (derivado del nombre del notebook)

# Hiperparametros de entrenamiento
BATCH_SIZE   = 32
LR           = 1e-4
EPOCHS_MAX   = 200
PATIENCE     = 12
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0

# DataLoaders con num_workers=0 (evita el AssertionError de Colab al destruir workers).
PIN_MEMORY  = (DEVICE.type == "cuda")
NUM_WORKERS = 0


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True   # reproducibilidad GPU
        torch.backends.cudnn.benchmark     = False


def evaluate(model, loader):
    model.eval()
    all_logits, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(DEVICE, non_blocking = True), A_norm).cpu().numpy()
            all_logits.append(logits.reshape(-1))
            all_y.append(yb.numpy().reshape(-1))
    logits_flat = np.concatenate(all_logits)
    y_flat      = np.concatenate(all_y)
    auc         = roc_auc_score(y_flat, logits_flat)
    return auc, logits_flat, y_flat


def save_checkpoint(ckpt_path, epoch, model_state, val_auc_best, history, seed):
    """Guarda el mejor modelo + metadata para sobrevivir a desconexiones de Colab."""
    torch.save({
        "epoch":         epoch,
        "model_state":   model_state,
        "val_auc_best":  val_auc_best,
        "history":       dict(history),
        "seed":          seed,
        "config": {
            "NUM_CHANNELS":      list(NUM_CHANNELS),
            "KERNEL_SIZE":       KERNEL_SIZE,
            "D_MODEL":           D_MODEL,
            "N_HEADS":           N_HEADS,
            "N_LAYERS":          N_LAYERS,
            "DROPOUT":           DROPOUT,
            "LOOKBACK":          LOOKBACK,
            "PREDICTION_WINDOW": PREDICTION_WINDOW,
            "n_features":        n_features,
            "n_active":          n_active,
        },
        "active_cells":     active_cells,
        "cell_region_arr":  cell_region_arr,
        "mean_train":       mean_train,
        "std_train":        std_train,
    }, ckpt_path)


def train_one_seed_transformer(seed, dropout = DROPOUT, verbose = True):
    set_all_seeds(seed)

    train_loader = DataLoader(
        train_ds, batch_size = BATCH_SIZE, shuffle = True,
        generator = torch.Generator().manual_seed(seed),
        num_workers = NUM_WORKERS, pin_memory = PIN_MEMORY,
    )
    val_loader   = DataLoader(val_ds,  batch_size = BATCH_SIZE, shuffle = False,
                              num_workers = NUM_WORKERS, pin_memory = PIN_MEMORY)
    test_loader  = DataLoader(test_ds, batch_size = BATCH_SIZE, shuffle = False,
                              num_workers = NUM_WORKERS, pin_memory = PIN_MEMORY)

    model = build_model(dropout)

    optimizer = torch.optim.Adam(model.parameters(), lr = LR, weight_decay = WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode = "max", factor = 0.5, patience = 5, min_lr = 1e-5,
    )
    import torch.nn.functional as F
    
    class FocalLoss(torch.nn.Module):
        def __init__(self, alpha=0.96, gamma=2.0):
            super().__init__()
            self.alpha = alpha
            self.gamma = gamma

        def forward(self, inputs, targets):
            bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
            pt = torch.exp(-bce_loss)
            alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
            focal_loss = alpha_t * ((1 - pt) ** self.gamma) * bce_loss
            return focal_loss.mean()

    criterion = FocalLoss(alpha=0.96, gamma=2.0)


    history = {"train_loss": [], "val_auc": []}
    best_val_auc, best_state, patience_counter = -1.0, None, 0

    ckpt_path = os.path.join(RESULTS_DIR, f"{NB_TAG}_ckpt_seed{seed}.pt")

    for epoch in range(1, EPOCHS_MAX + 1):
        model.train()
        train_losses = []

        if verbose:
            pbar = tqdm(
                train_loader,
                desc       = f"Epoch {epoch:>2}/{EPOCHS_MAX}",
                leave      = True,
                bar_format = "{desc}: {percentage:3.0f}%|{bar:25}| "
                             "{n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]{postfix}",
            )
        else:
            pbar = train_loader

        for xb, yb in pbar:
            xb = xb.to(DEVICE, non_blocking = True)
            yb = yb.to(DEVICE, non_blocking = True)
            optimizer.zero_grad()
            logits = model(xb, A_norm)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            train_losses.append(loss.item())
            if verbose:
                pbar.set_postfix_str(f"loss={np.mean(train_losses):.4f}")

        # Cerrar la barra antes de validar
        if verbose:
            pbar.close()

        val_auc, _, _ = evaluate(model, val_loader)
        scheduler.step(val_auc)

        history["train_loss"].append(np.mean(train_losses))
        history["val_auc"].append(val_auc)

        new_best = val_auc > best_val_auc
        if new_best:
            best_val_auc, best_state, patience_counter = val_auc, copy.deepcopy(model.state_dict()), 0
            # Checkpoint persistente al mejorar val_auc
            save_checkpoint(ckpt_path, epoch, best_state, best_val_auc, history, seed)
        else:
            patience_counter += 1

        if verbose:
            mark = " *" if new_best else ""
            tqdm.write(
                f"           val_auc={val_auc:.4f}   "
                f"best={best_val_auc:.4f}{mark}   "
                f"lr={optimizer.param_groups[0]['lr']:.0e}"
            )

        if patience_counter >= PATIENCE:
            if verbose:
                tqdm.write(f"Early stop tras epoch {epoch} (best val_auc={best_val_auc:.4f})")
            break

    # Restaurar mejor modelo y evaluar en test
    model.load_state_dict(best_state)
    test_auc, test_logits, test_y = evaluate(model, test_loader)

    return {
        "seed":         seed,
        "history":      history,
        "val_auc_best": best_val_auc,
        "test_auc":     test_auc,
        "test_logits":  test_logits,
        "test_y":       test_y,
        "ckpt_path":    ckpt_path,
    }


print(f"=== Configuracion de entrenamiento del Transformer ({NB_TAG}) ===")
print(f"DEVICE={DEVICE}  pin_memory={PIN_MEMORY}  num_workers={NUM_WORKERS}")
print(f"BATCH_SIZE={BATCH_SIZE}  LR={LR}  EPOCHS_MAX={EPOCHS_MAX}  PATIENCE={PATIENCE}")
print(f"WEIGHT_DECAY={WEIGHT_DECAY}  dropout={DROPOUT}  GRAD_CLIP={GRAD_CLIP}")
print(f"Loss: BCEWithLogits + pos_weight={POS_WEIGHT:.2f}")
print(f"Regiones: {REGIONS_TAG}  |  n_active={n_active}  |  d_model={D_MODEL}  heads={N_HEADS}")
print(f"Checkpoint dir: {RESULTS_DIR}/  (se guarda al mejorar val_auc)")
print()

In [ ]:
# --- Multi-seed: entrenar las 5 semillas y agregar (ensemble) ---
SEEDS = [13, 42, 123, 256, 1024]

multiseed_results = []
t0_ms = time.time()
for s in SEEDS:
    print(f">>> Seed {s} ...", flush=True)
    r = train_one_seed_transformer(seed=s, verbose=True)
    multiseed_results.append(r)
    print(f"    val_auc={r['val_auc_best']:.4f}   test_auc={r['test_auc']:.4f}", flush=True)
print(f"\nTiempo multi-seed total: {(time.time()-t0_ms)/60:.1f} min")

# El test set es el mismo para todas las seeds -> test_y compartido
test_y               = multiseed_results[0]["test_y"]
test_logits_per_seed = [r["test_logits"] for r in multiseed_results]
ensemble_logits      = np.mean(test_logits_per_seed, axis=0)   # ensemble = media de logits

test_aucs = [r["test_auc"] for r in multiseed_results]
print(f"\ntest_auc por seed: {[f'{a:.4f}' for a in test_aucs]}")
print(f"media +/- std:     {np.mean(test_aucs):.4f} +/- {np.std(test_aucs, ddof=1):.4f}")

# --- Curvas de entrenamiento de las 5 seeds (variabilidad + sobreajuste) ---
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
for r in multiseed_results:
    ep = np.arange(1, len(r["history"]["val_auc"]) + 1)
    axs[0].plot(ep, r["history"]["train_loss"], alpha=0.7, label=f"seed {r['seed']}")
    axs[1].plot(ep, r["history"]["val_auc"],    alpha=0.7, label=f"seed {r['seed']}")
axs[0].set_xlabel("Epoca"); axs[0].set_ylabel("Train loss")
axs[0].set_title("Perdida de entrenamiento (todas las seeds)"); axs[0].grid(alpha=0.3)
axs[1].axhline(0.5447, color="orange", ls=":", alpha=0.8, label="LogReg nb03 (0.545)")
axs[1].set_xlabel("Epoca"); axs[1].set_ylabel("Val AUC-ROC")
axs[1].set_title("Val AUC (todas las seeds)"); axs[1].legend(fontsize=8); axs[1].grid(alpha=0.3)
plt.suptitle(f"Transformer {NB_TAG} - {REGIONS_TAG} - multi-seed", fontweight="bold")
plt.tight_layout()
curvas_path = os.path.join(FIGURES_DIR, f"{NB_TAG}_{REGIONS_TAG.replace('+','_')}_curvas_multiseed.png")
plt.savefig(curvas_path, dpi=300, bbox_inches="tight"); plt.show()
print(f"Figura curvas multi-seed: {curvas_path}")


## 5. Evaluación

In [ ]:
# --- Evaluacion completa: metricas + Molchan + baseline Poisson + pickle global ---
import pickle
from sklearn.metrics import roc_curve
from utils.evaluacion import (metricas_completas, resumen_multiseed, ttest_una_muestra,
                              plot_molchan, baseline_poisson)

# 1) Metricas por seed -> media +/- std + IC95; y metricas del ensemble
metricas_por_seed = [metricas_completas(r["test_y"], r["test_logits"])
                     for r in multiseed_results]
resumen      = resumen_multiseed(metricas_por_seed)
met_ensemble = metricas_completas(test_y, ensemble_logits)

# 2) Baseline de Poisson (tasa base constante por celda)
y_test_p, score_p, p_cell = baseline_poisson(y_full, t_train, t_test)
met_poisson = metricas_completas(y_test_p, score_p)

# 3) Tabla comparativa COMPLETA (todas las metricas calculadas)
metricas_a_mostrar = [
    "roc_auc", "pr_auc",
    "precision@0.5%", "recall@0.5%", "lift@0.5%",
    "precision@1%",   "recall@1%",   "lift@1%",
    "precision@5%",   "recall@5%",   "lift@5%",
]
print(f"{'Metrica':16} | {'modelo (media+/-std)':22} | {'ensemble':9} | {'Poisson':8}")
print("-" * 66)
for k in metricas_a_mostrar:
    r = resumen[k]
    print(f"{k:16} | {r['mean']:7.4f} +/- {r['std']:.4f}    | "
          f"{met_ensemble[k]:>7.4f}  | {met_poisson[k]:>7.4f}")
print(f"(base_rate test = {met_ensemble['base_rate']:.4f}  ->  lift = precision / base_rate)")

# 4) Test estadistico: las 5 AUCs del modelo vs el AUC (escalar) de Poisson
tt = ttest_una_muestra([m["roc_auc"] for m in metricas_por_seed], met_poisson["roc_auc"])
sig = "significativo" if tt["pvalue"] < 0.05 else "NO significativo"
print(f"\nt-test ROC-AUC modelo vs Poisson:  t={tt['statistic']:.2f}  p={tt['pvalue']:.4g}  ({sig} al 5%)")

# 5) Diagrama de Molchan: ensemble vs Poisson (300 DPI)
# --- AUC temporal por celda (aísla la señal temporal de la espacial) ---
from utils.evaluacion import auc_temporal_por_celda
auc_t_modelo, aucs_t_modelo = auc_temporal_por_celda(test_y, ensemble_logits, n_active)
auc_t_poisson, _            = auc_temporal_por_celda(y_test_p, score_p, n_active)
print()
print("=== AUC TEMPORAL POR CELDA (lo que de verdad mide el precursor) ===")
print(f"  modelo (ensemble): {auc_t_modelo:.4f}   (Poisson: {auc_t_poisson:.4f}, =0.5 por construccion)")
print(f"  celdas evaluadas:  {len(aucs_t_modelo)} de {n_active} (>= 5 positivos en test)")
if len(aucs_t_modelo):
    print(f"  % celdas con AUC temporal > 0.5: {100*np.mean(aucs_t_modelo > 0.5):.1f}%")
    print("  -> >0.5 = senal precursora real ; ~0.5 = solo mapa espacial (climatologia)")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.hist(aucs_t_modelo, bins=20, color="darkred", alpha=0.75, edgecolor="white")
    ax.axvline(0.5, color="gray", ls="--", label="azar / Poisson (0.5)")
    ax.axvline(auc_t_modelo, color="black", ls="-", lw=2, label=f"media = {auc_t_modelo:.3f}")
    ax.set_xlabel("AUC temporal de la celda"); ax.set_ylabel("n de celdas")
    ax.set_title(f"AUC temporal por celda - {REGIONS_TAG} 2023-2024", fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
    auct_path = os.path.join(FIGURES_DIR, f"{NB_TAG}_{REGIONS_TAG.replace('+','_')}_auc_temporal.png")
    plt.savefig(auct_path, dpi=300, bbox_inches="tight"); plt.show()
    print(f"  Figura AUC temporal: {auct_path}")

fig, ax = plt.subplots(figsize=(6.5, 6.5))
plot_molchan(test_y,   ensemble_logits, ax=ax, label=f"Transformer {NB_TAG} (ensemble)", color="darkred")
plot_molchan(y_test_p, score_p,         ax=ax, label="Poisson (tasa base)",              color="gray")
ax.set_title(f"Diagrama de Molchan - {REGIONS_TAG} 2023-2024", fontweight="bold")
plt.tight_layout()
molchan_path = os.path.join(FIGURES_DIR, f"{NB_TAG}_{REGIONS_TAG.replace('+','_')}_molchan.png")
plt.savefig(molchan_path, dpi=300, bbox_inches="tight"); plt.show()
print(f"\nFigura Molchan: {molchan_path}")

# 6) ROC del ensemble vs Poisson (300 DPI)
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="Aleatorio (0.5)")
fpr, tpr, _ = roc_curve(test_y, ensemble_logits)
ax.plot(fpr, tpr, color="darkred", lw=2.2, label=f"Transformer ensemble (AUC={met_ensemble['roc_auc']:.4f})")
fpr_p, tpr_p, _ = roc_curve(y_test_p, score_p)
ax.plot(fpr_p, tpr_p, color="dimgray", lw=1.5, label=f"Poisson (AUC={met_poisson['roc_auc']:.4f})")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC test - {REGIONS_TAG} 2023-2024", fontweight="bold")
ax.legend(loc="lower right"); ax.grid(alpha=0.3); ax.set_aspect("equal")
plt.tight_layout()
roc_path = os.path.join(FIGURES_DIR, f"{NB_TAG}_{REGIONS_TAG.replace('+','_')}_roc_test.png")
plt.savefig(roc_path, dpi=300, bbox_inches="tight"); plt.show()
print(f"Figura ROC: {roc_path}")

# 7) Pickle COMPLETO para el notebook futuro de comparacion global entre modelos
TAG_MS = f"{NB_TAG}_{REGIONS_TAG.replace('+','_')}_multiseed"
results_multiseed = {
    "tag":                  TAG_MS,
    "nb_tag":               NB_TAG,
    "regions_tag":          REGIONS_TAG,
    "seeds":                SEEDS,
    "history_per_seed":     [r["history"] for r in multiseed_results],   # curvas (train_loss, val_auc) por epoca
    "test_y":               test_y,
    "test_logits_per_seed": test_logits_per_seed,
    "ensemble_logits":      ensemble_logits,
    "val_auc_per_seed":     [r["val_auc_best"] for r in multiseed_results],
    "test_auc_per_seed":    test_aucs,
    "metricas_por_seed":    metricas_por_seed,
    "metricas_ensemble":    met_ensemble,
    "poisson_score":        score_p,
    "metricas_poisson":     met_poisson,
    "auc_temporal_modelo":  auc_t_modelo,
    "aucs_temporal_celda":  aucs_t_modelo,
    "auc_temporal_poisson": auc_t_poisson,
}
pickle_path_ms = os.path.join(RESULTS_DIR, f"{TAG_MS}_results.pkl")
with open(pickle_path_ms, "wb") as f:
    pickle.dump(results_multiseed, f)
print(f"Pickle multi-seed (para comparacion global): {pickle_path_ms}")


In [ ]:
# === REGENERAR resultados 5-seed desde los CHECKPOINTS (NO re-entrena) ===
# Ejecutar ESTA celda en lugar de la de multi-seed: carga los 5 .pt, infiere sobre el
# test canonico y reconstruye el pickle + las figuras (ROC, Molchan, AUC temporal, curvas).
import torch, pickle
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve
from utils.evaluacion import (metricas_completas, resumen_multiseed, plot_molchan,
                              baseline_poisson, auc_temporal_por_celda)

SEEDS = [13, 42, 123, 256, 1024]
test_loader_r = DataLoader(test_ds, batch_size=64, shuffle=False)

logits_per_seed, histories, val_bests = [], [], []
print("Cargando checkpoints e infiriendo (sin entrenar)...")
for s in SEEDS:
    ck = torch.load(os.path.join(RESULTS_DIR, f"{NB_TAG}_ckpt_seed{s}.pt"),
                    map_location=DEVICE, weights_only=False)
    model = build_model()
    model.load_state_dict(ck["model_state"])
    model.eval()
    auc, logits, test_y = evaluate(model, test_loader_r)
    logits_per_seed.append(logits); histories.append(ck["history"])
    val_bests.append(float(ck["val_auc_best"]))
    print(f"  seed {s:>4}: test_auc={auc:.4f}  (val_best={ck['val_auc_best']:.4f})")

ensemble_logits = np.mean(logits_per_seed, axis=0)
test_aucs = [float(roc_auc_score(test_y, lg)) for lg in logits_per_seed]
print(f"\ntest_auc media +/- std: {np.mean(test_aucs):.4f} +/- {np.std(test_aucs, ddof=1):.4f}")

# --- Metricas + Poisson + AUC temporal ---
metricas_por_seed = [metricas_completas(test_y, lg) for lg in logits_per_seed]
resumen = resumen_multiseed(metricas_por_seed)
met_ensemble = metricas_completas(test_y, ensemble_logits)
y_test_p, score_p, p_cell = baseline_poisson(y_full, t_train, t_test)
met_poisson = metricas_completas(y_test_p, score_p)
auc_t_modelo, aucs_t = auc_temporal_por_celda(test_y, ensemble_logits, n_active)
auc_t_poisson, _ = auc_temporal_por_celda(y_test_p, score_p, n_active)

print(f"\n{'Metrica':14} | modelo media+/-std | ensemble | Poisson")
for k in ["roc_auc","pr_auc","lift@1%"]:
    r=resumen[k]
    print(f"  {k:12} | {r['mean']:.4f}+/-{r['std']:.4f} | {met_ensemble[k]:.4f}  | {met_poisson[k]:.4f}")
print(f"AUC temporal: modelo={auc_t_modelo:.4f}  Poisson={auc_t_poisson:.4f}  (base_rate={met_ensemble['base_rate']:.4f})")

# --- Guardar pickle correcto ---
TAG_MS = f"{NB_TAG}_{REGIONS_TAG.replace('+','_')}_multiseed"
results_multiseed = {
    "tag": TAG_MS, "nb_tag": NB_TAG, "regions_tag": REGIONS_TAG, "seeds": SEEDS,
    "history_per_seed": histories, "test_y": test_y,
    "test_logits_per_seed": logits_per_seed, "ensemble_logits": ensemble_logits,
    "val_auc_per_seed": val_bests, "test_auc_per_seed": test_aucs,
    "metricas_por_seed": metricas_por_seed, "metricas_ensemble": met_ensemble,
    "poisson_score": score_p, "metricas_poisson": met_poisson,
    "auc_temporal_modelo": auc_t_modelo, "aucs_temporal_celda": aucs_t,
    "auc_temporal_poisson": auc_t_poisson,
}
pkl = os.path.join(RESULTS_DIR, f"{TAG_MS}_results.pkl")
with open(pkl, "wb") as f: pickle.dump(results_multiseed, f)
print(f"\nPickle regenerado: {pkl}")

# --- Figuras (ROC, Molchan, AUC temporal, curvas) ---
fig, ax = plt.subplots(figsize=(7,7))
ax.plot([0,1],[0,1],"--",color="gray",lw=1,label="Aleatorio (0.5)")
fpr,tpr,_=roc_curve(test_y, ensemble_logits); ax.plot(fpr,tpr,color="darkred",lw=2.2,label=f"Transformer ensemble (AUC={met_ensemble['roc_auc']:.4f})")
fpr,tpr,_=roc_curve(y_test_p, score_p); ax.plot(fpr,tpr,color="dimgray",lw=1.5,label=f"Poisson (AUC={met_poisson['roc_auc']:.4f})")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title(f"ROC test - {REGIONS_TAG} 2023-2024", fontweight="bold")
ax.legend(loc="lower right"); ax.grid(alpha=0.3); ax.set_aspect("equal"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{NB_TAG}_roc_test.png"), dpi=300, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(6.5,6.5))
plot_molchan(test_y, ensemble_logits, ax=ax, label="Transformer (ensemble)", color="darkred")
plot_molchan(y_test_p, score_p, ax=ax, label="Poisson", color="gray")
ax.set_title(f"Diagrama de Molchan - {REGIONS_TAG}", fontweight="bold"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{NB_TAG}_molchan.png"), dpi=300, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(7,4.5))
ax.hist(aucs_t, bins=20, color="darkred", alpha=0.75, edgecolor="white")
ax.axvline(0.5, color="gray", ls="--", label="azar / Poisson (0.5)")
ax.axvline(auc_t_modelo, color="black", lw=2, label=f"media={auc_t_modelo:.3f}")
ax.set_xlabel("AUC temporal de la celda"); ax.set_ylabel("n de celdas")
ax.set_title(f"AUC temporal por celda - {REGIONS_TAG}", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{NB_TAG}_auc_temporal.png"), dpi=300, bbox_inches="tight"); plt.show()

fig, axs = plt.subplots(1,2,figsize=(14,5))
for i,h in enumerate(histories):
    ep=np.arange(1,len(h["val_auc"])+1)
    axs[0].plot(ep,h["train_loss"],alpha=0.7,label=f"seed {SEEDS[i]}")
    axs[1].plot(ep,h["val_auc"],alpha=0.7,label=f"seed {SEEDS[i]}")
axs[0].set_xlabel("Epoca"); axs[0].set_ylabel("Train loss"); axs[0].set_title("Perdida (5 seeds)"); axs[0].grid(alpha=0.3)
axs[1].set_xlabel("Epoca"); axs[1].set_ylabel("Val AUC"); axs[1].set_title("Val AUC (5 seeds)"); axs[1].legend(fontsize=8); axs[1].grid(alpha=0.3)
plt.suptitle(f"Curvas de entrenamiento - Transformer {NB_TAG}", fontweight="bold"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{NB_TAG}_curvas_multiseed.png"), dpi=300, bbox_inches="tight"); plt.show()
print("Figuras 05 regeneradas (ROC, Molchan, AUC temporal, curvas).")
